<a href="https://colab.research.google.com/github/MuhammadEhtisham776/flyrank-ml-internship-starter/blob/main/Copy_of_w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import json, os

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

print("HF token loaded:", HF_TOKEN is not None)

HF token loaded: True


# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadEhtisham776/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Logistic Regression first, Random Forest second — both evaluated as a ranking via precision@K.**

The lane's decision is "which ones first?" — per the toolkit, that means any classifier's
probability, scored at precision@K, not a bare accuracy number. The label is observed and binary
(worse than average vs. not), which points to Logistic Regression as the readable starting point,
then Random Forest to see whether the extra complexity actually earns its keep.

**Label:** `y = 1` if a page's `ctr_gap` sits in the worst 10% of the distribution — a *pure*
gap-based definition, deliberately different from what the Week-4 baseline score optimizes (which
multiplies gap by `impressions`). That keeps this comparison honest rather than rigged: the
baseline isn't being tested against the exact thing it was built to maximize.

**Features:** the same structural, non-label signals from Weeks 2-4 — `position_tier`,
`word_count`, `content_type`, `main_intent`, `search_volume`, `impressions`. Never `ctr` or
`ctr_gap` themselves — the model has to infer likely underperformance from context, not read it
off the label.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np
import json, os
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance

REPO = "hf://datasets/FlyRank/internship-warehouse"
os.makedirs("work/outputs", exist_ok=True)

fact = pd.read_parquet(f"{REPO}/fact_content_daily_performance/month=2026-03/data_0.parquet")
dim_content = pd.read_parquet(f"{REPO}/dim_content.parquet")

fact_avail = fact[fact["gsc_data_available"] == True].copy()
agg = (fact_avail.groupby(["client_hash_id", "content_hash_id"])
       .agg(impressions=("gsc_impressions", "sum"),
            clicks=("gsc_clicks", "sum"),
            sum_position=("gsc_sum_position", "sum"))
       .reset_index())
agg["avg_position"] = agg["sum_position"] / agg["impressions"]
agg["ctr"] = agg["clicks"] / agg["impressions"] * 100

VISIBLE_MIN_IMPR = 150
visible = agg[(agg["impressions"] >= VISIBLE_MIN_IMPR) & (agg["avg_position"] > 0)].copy()

bins, labels = [0, 3, 10, 20, 50, np.inf], ["top_3", "page_1", "striking", "page_3_5", "deep"]
visible["position_tier"] = pd.cut(visible["avg_position"], bins=bins, labels=labels)
visible["tier_median_ctr"] = visible.groupby("position_tier", observed=True)["ctr"].transform("median")
visible["ctr_gap"] = visible["ctr"] - visible["tier_median_ctr"]

content_cols = ["client_hash_id", "content_hash_id", "content_type", "main_intent", "word_count", "search_volume"]
visible = visible.merge(dim_content[content_cols], on=["client_hash_id", "content_hash_id"], how="left")

# Week 4 baseline score, unchanged
visible["underperforming"] = (visible["ctr_gap"] < 0).astype(int)
visible["baseline_score"] = visible["underperforming"] * (-visible["ctr_gap"]) * visible["impressions"]

# true label: worst decile of ctr_gap alone (NOT impression-weighted, unlike the baseline score)
threshold = visible["ctr_gap"].quantile(0.10)
visible["y_true"] = (visible["ctr_gap"] <= threshold).astype(int)
print("label threshold (10th pct of ctr_gap):", threshold)
print("positive rate:", visible["y_true"].mean())


label threshold (10th pct of ctr_gap): -0.2012650948821162
positive rate: 0.14032226498793138


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by `client_hash_id`.** Pages from the same client share templates, niches, and CMS
quirks — a random row-level split would let the model partly memorize client identity through
correlated structural features, inflating the score without teaching anything that generalizes to
a *new* client. `GroupShuffleSplit` guarantees zero client overlap between train and test, so the
test score reflects "does this hold for clients the model has never seen," which is the honest
version of the question.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(visible, groups=visible["client_hash_id"]))
train_df = visible.iloc[train_idx].copy()
test_df = visible.iloc[test_idx].copy()
print("train rows:", len(train_df), "| clients:", train_df["client_hash_id"].nunique())
print("test rows:", len(test_df), "| clients:", test_df["client_hash_id"].nunique())
print("client overlap train/test:", len(set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])))

train rows: 85429 | clients: 31
test rows: 6545 | clients: 11
client overlap train/test: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

FEATURES = ["position_tier", "word_count", "content_type", "main_intent", "search_volume", "impressions"]
cat_cols = ["position_tier", "content_type", "main_intent"]
num_cols = ["word_count", "search_volume", "impressions"]

X_train, X_test = train_df[FEATURES].copy(), test_df[FEATURES].copy()
y_train, y_test = train_df["y_true"], test_df["y_true"]
for c in num_cols:
    X_train[c] = X_train[c].fillna(X_train[c].median())
    X_test[c] = X_test[c].fillna(X_train[c].median())
for c in cat_cols:
    X_train[c] = X_train[c].astype(str).fillna("missing")
    X_test[c] = X_test[c].astype(str).fillna("missing")

pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])

logreg = Pipeline([("prep", pre), ("model", LogisticRegression(max_iter=1000, random_state=42))])
logreg.fit(X_train, y_train)
proba_lr = logreg.predict_proba(X_test)[:, 1]

rf = Pipeline([("prep", pre), ("model", RandomForestClassifier(
    n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=42, n_jobs=-1))])
rf.fit(X_train, y_train)
proba_rf = rf.predict_proba(X_test)[:, 1]

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return y_true.values[order].mean()

rows = []
for name, scores in [("baseline_rule (Wk4)", test_df["baseline_score"].values),
                      ("logistic_regression", proba_lr),
                      ("random_forest", proba_rf)]:
    rows.append({"model": name,
                 "precision_at_50": round(precision_at_k(y_test, scores, 50), 3),
                 "precision_at_100": round(precision_at_k(y_test, scores, 100), 3),
                 "roc_auc": round(roc_auc_score(y_test, scores), 3)})
base_rate = round(y_test.mean(), 3)
rows.append({"model": "random_baseline (base rate)", "precision_at_50": base_rate,
             "precision_at_100": base_rate, "roc_auc": 0.5})

comparison = pd.DataFrame(rows)
print("=== MODEL vs BASELINE (same test split, same metric) ===")
print(comparison.to_string(index=False))


=== MODEL vs BASELINE (same test split, same metric) ===
                      model  precision_at_50  precision_at_100  roc_auc
        baseline_rule (Wk4)            0.160             0.200    0.885
        logistic_regression            0.480             0.520    0.799
              random_forest            0.460             0.550    0.790
random_baseline (base rate)            0.188             0.188    0.500


At the top of the queue (precision@50/@100), both models beat the baseline by a wide margin — and
the baseline is actually *worse than random guessing* at precision@50 (0.160 vs. a 0.188 base
rate). At overall ranking quality (ROC-AUC across the full score range), the baseline wins clearly
(0.885 vs. ~0.79-0.80).

Both halves are real and both make sense. The baseline's score is `gap × impressions`, so at the
very top of its ranked list, high-impression pages with only a *moderate* gap can out-rank a
low-impression page with a *much worse* gap — exactly the "row 10" pattern flagged as the weakest
pick in Week 4's baseline. That distortion is worst right at the top (K=50), which is exactly where
an editor would actually be working. Across the *whole* score range, though, the baseline still
tracks the true gap reasonably well overall (hence the high AUC) — it's specifically the top-of-queue
ordering that breaks.

The models, restricted to structural features only, can't see the exact gap value, so they can't
game the top of the list the same way — and that turns out to matter more for this decision than
raw discriminative power across the whole range. **For the actual decision this lane supports
(a fixed-size weekly review queue), precision@K is the metric that matters, and both models beat
the baseline on it.**

**Random Forest doesn't earn its complexity here.** Logistic Regression and Random Forest are
within a few points of each other on every metric (precision@50: 0.480 vs 0.460; precision@100:
0.520 vs 0.550; AUC: 0.799 vs 0.790) — essentially tied, with no consistent winner. Per "add
complexity only when the comparison earns it," Logistic Regression is the better pick to carry
forward: same performance, and its coefficients are directly readable.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42, scoring="roc_auc")
imp_df = pd.DataFrame({"feature": FEATURES,
                        "importance_mean": perm.importances_mean,
                        "importance_std": perm.importances_std}).sort_values("importance_mean", ascending=False)
print("=== permutation importance (random forest) ===")
print(imp_df.to_string(index=False))

test_df = test_df.copy()
test_df["pred_proba_rf"] = proba_rf
test_df["pred_label"] = (test_df["pred_proba_rf"] >= 0.5).astype(int)

fn = test_df[(test_df["y_true"] == 1) & (test_df["pred_label"] == 0)].sort_values("impressions", ascending=False).head(3)
fp = test_df[(test_df["y_true"] == 0) & (test_df["pred_label"] == 1)].sort_values("pred_proba_rf", ascending=False).head(3)
err_cols = ["client_hash_id", "content_hash_id", "position_tier", "ctr_gap", "impressions", "word_count", "content_type", "pred_proba_rf"]

print("\n=== 3 false negatives (missed true priority pages) ===")
print(fn[err_cols].to_string(index=False))
print("\n=== 3 false positives (flagged, not actually true priority) ===")
print(fp[err_cols].to_string(index=False))

with open("work/outputs/w05_metrics.json", "w") as f:
    json.dump({"comparison_table": comparison.to_dict(orient="records"),
               "permutation_importance": imp_df.to_dict(orient="records"),
               "label_threshold": float(threshold), "positive_rate": float(y_test.mean())}, f, indent=2)
print("\nwrote work/outputs/w05_metrics.json")


=== permutation importance (random forest) ===
      feature  importance_mean  importance_std
position_tier         0.197506        0.004544
  impressions         0.130943        0.004978
   word_count         0.006701        0.001263
  main_intent         0.001545        0.000601
 content_type         0.001223        0.000529
search_volume         0.000208        0.000419

=== 3 false negatives (missed true priority pages) ===
         client_hash_id          content_hash_id position_tier   ctr_gap  impressions  word_count    content_type  pred_proba_rf
client_1a730cb2640a1abf content_d61fc394d10cba41         top_3 -0.217390        38000      2484.0 keyword article       0.071412
client_1a730cb2640a1abf content_b9d46abd9ffa6c8a        page_1 -0.201265        15902      2642.0 keyword article       0.068034
client_c182d11e4862a37d content_d2eb49b1f5f3fa34        page_1 -0.201265        14482         NaN keyword article       0.116043

=== 3 false positives (flagged, not actually true p

**What the model leans on:** `position_tier` (0.198) and `impressions` (0.131) dominate;
`word_count`, `main_intent`, `content_type`, `search_volume` barely register (all under 0.007).
`position_tier` mattering this much is *not* suspicious even though the label is tier-normalized —
it makes sense structurally: `deep`-tier CTR sits at a near-zero floor, so extremely negative gaps
can only really occur at `top_3`/`page_1`, where the tier median is high enough to leave room to
fall far below it. `search_volume` landing at essentially zero importance lines up with Week 4's
`OPPOSITE` signal-check verdict — a second, independent confirmation that raw keyword demand isn't
a useful signal for this label.

**Where it's wrong:** the 3 false negatives are all borderline — their `ctr_gap` (-0.217, -0.201,
-0.201) sits right at or barely past the -0.2013 threshold, so these are near-miss calibration
errors, not egregious ones, though they're exactly the high-impression (14k-38k) pages an editor
would most want caught. The 3 false positives are the opposite kind of mistake: very low impressions
(176-238) and *positive* gaps (well above their tier's median, i.e. genuinely strong performers,
not weak ones) — the model seems to read "unusual/extreme relative to typical volume" as risk
without reliably telling direction apart at low impression counts, and 2 of the 3 are missing
`word_count`, echoing the same missing-field pattern flagged in Week 4's baseline review.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.